# 🌱 GA Potager — atelier data scientist

Workflow du notebook :
1. **Configurer le bac** dans le bloc de code ci-dessous (dimensions, terre, substrats…)
2. **Lancer l'algorithme génétique** (la lib Rust via le CLI en mode `--json`)
3. **Lire le gagnant** et *pourquoi* il gagne (décomposition du score)
4. **Explorer toute la population** : les bonnes propositions, et les moins bonnes

## 1. Configuration du bac

Modifie librement les valeurs, puis réexécute cette cellule (et les suivantes).

In [ ]:
import json
from pathlib import Path

# --- Ton bac, à toi de jouer ---
BAC = {
    "id": "reference",
    "nom": "Bac de référence",
    "dimensions_cm": {"longueur": 120, "largeur": 80, "hauteur": 50},
    "hauteur_terre_cm": 40,
    "marge_bord_cm": 5,
    "protection": "exterieur",
    "substrats": ["terreau", "compost", "fibre_de_coco"],
    "compost_disponible": True,
    "drainage_eau": True,
    "exposition": "plein_soleil",
    "saison_cible": "ete",
    "temp_moyenne_c": {"min": 15, "max": 30},
}

# Où vit le repo (le notebook est dans notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

# Écrit bacs.json : c'est L'entrée de l'AG
(DATA / "bacs.local.json").write_text(
    json.dumps([BAC], indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"bacs.local.json écrit (hors git) : {BAC['nom']}")

### 📖 Référence des paramètres du bac (valeurs exactes acceptées)

Source de vérité : les enums Rust de `models/` (une valeur mal orthographiée = erreur de parsing au lancement). Les chaînes ci-dessous sont **exactement** ce que le JSON attend.

| Paramètre | Type | Valeurs possibles | Signification |
|---|---|---|---|
| `id` | texte | libre, unique | identifiant du bac (ex. `"reference"`) |
| `nom` | texte | libre | nom affiché |
| `dimensions_cm` | objet | `{"longueur": n, "largeur": n, "hauteur": n}` (cm) | dimensions extérieures |
| `hauteur_terre_cm` | nombre | ≥ 0 | profondeur de terre ; les racines des plantes doivent tenir dedans (sauf bac hydro) |
| `marge_bord_cm` | nombre | ≥ 0 | bande non plantable sur chaque bord ; la surface utile = (L−2·marge)×(l−2·marge) |
| `substrats` | liste | chaînes libres : `"terreau"`, `"compost"`, `"fibre_de_coco"`, `"sableux"`… | substrats disponibles ; **vide + drainage = bac hydroponique** (contraintes de racines/substrat neutralisées) |
| `compost_disponible` | booléen | `true` / `false` | compost disponible pour ce bac |
| `drainage_eau` | booléen | `true` / `false` | le bac évacue l'eau ; permet les plantes à `enracinement_eau` (hydroponie, aquaponie plus tard) |
| `exposition` | enum | `"plein_soleil"`, `"mi_ombre"`, `"ombre"` | ensoleillement du bac |
| `saison_cible` | enum | `"printemps"`, `"ete"`, `"automne"`, `"hiver"` | saison visée ; les plantes doivent l'accepter (contrainte dure) |
| `temp_moyenne_c` | objet | `{"min": n, "max": n}` (°C) | plage de températures du bac ; doit chevaucher la plage de survie des plantes |
| `protection` | enum | `"exterieur"`, `"serre_froide"`, `"serre_chauffee"` | abri du bac — **c'est ici que vont "serre ensoleillée", "serre chaude"** etc. : `exterieur` = plein air, `serre_froide` = abri non chauffé (tunnel, véranda), `serre_chauffee` = abri chauffé |

💡 **Exemples** : "serre ensoleillée" → `protection: "serre_froide"` + `exposition: "plein_soleil"` ; "serre chaude" → `protection: "serre_chauffee"` ; "bac d'intérieur devant baie vitrée" → `"exterieur"` + `"plein_soleil"` (le modèle ne distingue pas encore intérieur/extérieur — à enrichir si besoin).

<details><summary>📖 Les mêmes valeurs côté <code>plantes.json</code> (pour tes enrichissements)</summary>

| Champ | Valeurs possibles |
|---|---|
| `taux_croissance` | `"lente"`, `"moyenne"`, `"invasive"` (invasive : interdite si surface utile < `seuil_invasif_cm2`) |
| `saison` | liste parmi `"printemps"`, `"ete"`, `"automne"`, `"hiver"` |
| `exposition` | `"plein_soleil"`, `"mi_ombre"`, `"ombre"` |
| `besoin_eau`, `besoins_nutriments.N/P/K` | `"faible"`, `"moyen"`, `"eleve"` (clés N/P/K en majuscules) |
| `enracinement_eau` | `true` = exige un bac à drainage |
| `temperature_c` | `min_survie` / `optimal_min` / `optimal_max` / `max_survie` — seule la survie est une contrainte dure |
| `rendement` | objet : `valeur` (kg/m²), `unite`, `duree_jours`, `source`, `niveau_confiance` |

</details>

## 2. Lancement de l'algorithme génétique

Le binaire Rust est compilé si besoin, puis lancé en mode `--json`. Change `SEED` pour explorer la variabilité des runs.

In [ ]:
import subprocess, sys

SEED = 42  # change pour explorer la variabilité

BIN = ROOT / "ga-plantes" / "target" / "debug" / ("ga-potager-cli.exe" if sys.platform == "win32" else "ga-potager-cli")
if not BIN.exists():
    print("Compilation du binaire...")
    subprocess.run(["cargo", "build", "--manifest-path", str(ROOT / "ga-plantes" / "Cargo.toml")], check=True)

proc = subprocess.run(
    [str(BIN), "--data-dir", str(DATA), "--config-path", str(DATA / "config.json"), "--seed", str(SEED), "--json", "--bacs", str(DATA / "bacs.local.json")],
    capture_output=True, text=True, encoding="utf-8", errors="replace",
)
if proc.returncode != 0:
    print(proc.stderr)  # erreurs de données : la validation explique quoi corriger
    raise SystemExit(1)

resultat = json.loads(proc.stdout[proc.stdout.index("{"):])
print(f"Évolution terminée en {resultat['generations']} générations (seed {resultat['seed']})")

## 3. Le gagnant — et *pourquoi* il gagne

Le score total est la somme de trois composantes pondérées. Le graphe montre laquelle porte la solution.

In [ ]:
import matplotlib.pyplot as plt

meilleur = resultat["meilleur"]
print("🏆 Potager gagnant — bac", meilleur["bac"])
for p in meilleur["plantes"]:
    print(f"   - {p['nom']} ({p['famille']})")
print(f"Score total : {meilleur['score']:.2f}")
print("\nInteractions appliquées :")
for i in meilleur["detail"]["interactions_actives"] or ["(aucune)"]:
    print(f"   {i}")

d = meilleur["detail"]
composantes = {"Rendement (x5)": d["rendement"], "Interactions (x7)": d["interactions"], "Occupation (x10)": d["occupation"]}
couleurs = ["#4caf50" if v >= 0 else "#e53935" for v in composantes.values()]

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(list(composantes.keys()), list(composantes.values()), color=couleurs)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"Pourquoi ce potager gagne — décomposition du score (total {meilleur['score']:.2f})")
for i, v in enumerate(composantes.values()):
    ax.text(v, i, f" {v:+.2f}", va="center")
plt.tight_layout(); plt.show()

## 4. Toute la population — les bonnes et les moins bonnes propositions

L'AG ne produit pas qu'un gagnant : la dernière génération contient toutes les solutions évaluées. L'histogramme montre la distribution, le classement les extrêmes.

In [ ]:
pop = sorted(resultat["population"], key=lambda x: -x["score"])
scores = [x["score"] for x in pop]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(scores, bins=20, color="#66bb6a", edgecolor="white")
ax.axvline(meilleur["score"], color="#1b5e20", linestyle="--", label=f"gagnant ({meilleur['score']:.1f})")
ax.set_xlabel("Score"); ax.set_ylabel("Nombre d'individus")
ax.set_title(f"Distribution des {len(pop)} individus de la population finale")
ax.legend(); plt.tight_layout(); plt.show()

print("🥇 Top 3 :")
for x in pop[:3]:
    print(f"   {x['score']:8.2f}  {' + '.join(x['plantes'])}")
print("\n💀 Flop 3 :")
for x in pop[-3:]:
    print(f"   {x['score']:8.2f}  {' + '.join(x['plantes']) or '(bac vide)'}")

## 🗺️ Le gagnant, vu du dessus

Schéma du bac (échelle réelle) : chaque plante est un cercle de rayon = la moitié de son espacement minimum. Si des cercles se chevauchent ou sortent du cadre, la combinaison est à la limite de la contrainte de surface — exactement ce que la validation et la fitness arbitrent.

In [ ]:
import json as _json
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

# Espacements réels depuis plantes.json
_plantes_data = _json.loads((DATA / "plantes.json").read_text(encoding="utf-8"))
espacements = {p["id"]: p["espacement_min_cm"] for p in _plantes_data}
familles = {p["id"]: p["famille"] for p in _plantes_data}

L, l = BAC["dimensions_cm"]["longueur"], BAC["dimensions_cm"]["largeur"]
marge = BAC["marge_bord_cm"]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.add_patch(Rectangle((0, 0), L, l, fill=False, edgecolor="black", linewidth=2))
ax.add_patch(Rectangle((marge, marge), L - 2*marge, l - 2*marge, fill=False, edgecolor="#2e7d32", linewidth=1, linestyle="--"))

COULEURS = {"Solanacées": "#e53935", "Fabacées": "#43a047", "Brassicacées": "#8e24aa", "Amaryllidacées": "#795548", "Lamiacées": "#00897b", "Apiacées": "#fb8c00", "Astéracées": "#fdd835", "Cucurbitacées": "#5e35b1"}

# Placement en rangées DANS la surface utile : chaque plante occupe un
# carré de côté = son espacement ; l'avance verticale entre rangées est
# la hauteur de la rangée précédente (sans double-comptage).
x, y = marge, marge
h_rangee = 0
deborde = []
for p in meilleur["plantes"]:
    esp = espacements[p["id"]]
    r = esp / 2
    if x + esp > L - marge:            # nouvelle rangée
        x = marge
        y += h_rangee                   # collée sous la précédente
        h_rangee = 0
    if y + esp > l - marge:            # déborderait du bas : on le signale
        deborde.append(p["id"])
        continue
    ax.add_patch(Circle((x + r, y + r), r, color=COULEURS.get(familles[p["id"]], "gray"), alpha=0.7))
    # label sous le cercle pour les petites plantes (illisibles au centre)
    if esp >= 30:
        ax.text(x + r, y + r, p["id"], ha="center", va="center", fontsize=9)
    else:
        ax.text(x + r, y - 1.5, p["id"], ha="center", va="top", fontsize=7)
    x += esp
    h_rangee = max(h_rangee, esp)

if deborde:
    print(f"⚠️ Placement illustratif incomplet (rangées trop serrées pour tout caser) : {', '.join(deborde)}")
    print("La combinaison reste VALIDE : la contrainte du Rust est la somme des carrés d'espacement ≤ surface utile. Ce dessin est indicatif, pas un plan de plantation.")

ax.set_xlim(-5, L + 5); ax.set_ylim(-5, l + 5)
ax.set_aspect("equal")
ax.set_title(f"Potager gagnant — bac {L}x{l} cm (schéma indicatif, échelle réelle)")
plt.tight_layout(); plt.show()

## 📈 L'évolution au fil des générations

**Convergence** : min / moyenne / max de la fitness à chaque génération (population initiale incluse en génération 0).
- `max` qui monte encore à la fin → augmente `generations_max` ou la `patience`
- `max` stagnant dès la génération 2-3 → population trop petite ou espace de recherche trop simple
- `min` qui remonte vers la `moyenne` → la sélection homogénéise la population (normal, mais trop tôt = perte de diversité)

In [ ]:
h = resultat["historique"]
gens = [x["generation"] for x in h]

fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(gens, [x["fitness_min"] for x in h], [x["fitness_max"] for x in h], alpha=0.15, color="#66bb6a", label="éventail min-max")
ax.plot(gens, [x["fitness_max"] for x in h], color="#1b5e20", linewidth=2, label="max")
ax.plot(gens, [x["fitness_moyenne"] for x in h], color="#66bb6a", linewidth=1.5, label="moyenne")
ax.plot(gens, [x["fitness_min"] for x in h], color="#a5d6a7", linewidth=1, label="min")
ax.set_xlabel("Génération"); ax.set_ylabel("Fitness")
ax.set_title("Convergence de la population")
ax.legend(); plt.tight_layout(); plt.show()

### Qui domine la population, et quand ?

% d'individus contenant chaque plante, génération après génération. C'est ici qu'on voit si une plante est bonne **dès le tirage initial** (courbe haute dès la gen 0) ou si elle **émerge** grâce à la sélection/mutation (courbe qui monte). Une plante qui décline était sur-représentée au départ mais perd le combat de la fitness.

In [ ]:
ids = sorted({pid for x in h for pid, _ in x["presence_plantes"]})
pop_size = len(resultat["population"])

fig, ax = plt.subplots(figsize=(8, 4.5))
for pid in ids:
    serie = []
    for x in h:
        counts = dict(x["presence_plantes"])
        serie.append(100 * counts.get(pid, 0) / pop_size)
    star = pid == "tomate"  # mets ici l'id que tu veux suivre
    ax.plot(gens, serie, label=pid, linewidth=2.5 if star else 1.2, alpha=1.0 if star else 0.6)

ax.set_xlabel("Génération"); ax.set_ylabel("% d'individus contenant la plante")
ax.set_title("Composition de la population au fil des générations")
ax.legend(ncol=2, fontsize=8); plt.tight_layout(); plt.show()

---

# 🌍 M4 — Robustesse multi-environnements

Le v1 optimisait pour UN bac. Ici, on lance l'AG sur **plusieurs environnements** (variantes du bac de référence) × **plusieurs seeds**, et on compare :
- le **meilleur score** atteint par environnement (la solution "robuste" se détache-t-elle ?)
- la **composition des potagers gagnants** (les mêmes plantes partout, ou des spécialistes par environnement ?)

⚠️ Attendu pédagogique : aucun plante du dataset n'a `hiver` dans ses saisons → le bac hiver ne produira **aucune combinaison valide**. C'est le système qui te dit d'enrichir les données (ou de changer la saison), pas un bug.

In [ ]:
import copy

# Variantes d'environnements, dérivées du bac de référence configuré en haut
def variantes_bac():
    v = {"reference": copy.deepcopy(BAC)}

    hiver = copy.deepcopy(BAC)
    hiver.update({"id": "hiver", "nom": "Bac hiver", "protection": "serre_froide",
                  "saison_cible": "hiver", "temp_moyenne_c": {"min": 5, "max": 15}})
    v["hiver"] = hiver

    hydro = copy.deepcopy(BAC)
    hydro.update({"id": "hydro", "nom": "Bac hydroponique", "hauteur_terre_cm": 0,
                  "substrats": [], "drainage_eau": True, "protection": "serre_chauffee",
                  "temp_moyenne_c": {"min": 20, "max": 30}})
    v["hydro"] = hydro

    balcon = copy.deepcopy(BAC)
    balcon.update({"id": "balcon", "nom": "Petit balcon",
                   "dimensions_cm": {**BAC["dimensions_cm"], "longueur": 60, "largeur": 40},
                   "marge_bord_cm": 3, "compost_disponible": False,
                   "saison_cible": "printemps", "temp_moyenne_c": {"min": 10, "max": 20}})
    v["balcon"] = balcon
    return v

def run_ag(bac, seed):
    """Un run complet de l'AG pour un bac et une seed donnés."""
    (DATA / "bacs.local.json").write_text(
        json.dumps([bac], indent=2, ensure_ascii=False), encoding="utf-8")
    proc = subprocess.run(
        [str(BIN), "--data-dir", str(DATA), "--config-path", str(DATA / "config.json"),
         "--seed", str(seed), "--json", "--bacs", str(DATA / "bacs.local.json")],
        capture_output=True, text=True, encoding="utf-8", errors="replace")
    if proc.returncode != 0:
        return None  # ex. : aucune plante compatible avec cet environnement
    return json.loads(proc.stdout[proc.stdout.index("{"):])

SEEDS = [42, 7, 1]  # plusieurs seeds : un seul run ne prouve rien
resultats_env = {}
for env_id, bac in variantes_bac().items():
    runs = [r for r in (run_ag(bac, s) for s in SEEDS) if r is not None]
    resultats_env[env_id] = runs
    if not runs:
        print(f"❌ {env_id} : aucune combinaison valide (aucune plante compatible avec cet environnement)")
    else:
        scores = [r["meilleur"]["score"] for r in runs]
        print(f"✅ {env_id} : meilleur score moyen {sum(scores)/len(scores):.2f} (min {min(scores):.2f}, max {max(scores):.2f}) sur {len(runs)} seeds")

In [ ]:
# Comparaison graphique : meilleur score par environnement (moyenne ± écart aux seeds)
envs = [e for e, runs in resultats_env.items() if runs]
moyennes = [sum(r["meilleur"]["score"] for r in resultats_env[e]) / len(resultats_env[e]) for e in envs]
mins = [min(r["meilleur"]["score"] for r in resultats_env[e]) for e in envs]
maxs = [max(r["meilleur"]["score"] for r in resultats_env[e]) for e in envs]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(envs, moyennes, color="#66bb6a", edgecolor="white")
yerr=[[max(0.0, m - mn) for m, mn in zip(moyennes, mins)],  # clamp : arrondi flottant peut rendre mean-min < 0\n                                   [max(0.0, mx - m) for m, mx in zip(moyennes, maxs)]],\n            fmt="none", ecolor="#1b5e20", capsize=5, label="écart entre seeds")
ax.set_ylabel("Meilleur score (moyenne sur les seeds)")
ax.set_title(f"Robustesse : l'AG sur {len(envs)} environnements × {len(SEEDS)} seeds")
ax.legend(); plt.tight_layout(); plt.show()

# Le potager gagnant de chaque environnement (meilleur run)
print("\n🥇 Potager gagnant par environnement :")
for e in envs:
    best_run = max(resultats_env[e], key=lambda r: r["meilleur"]["score"])
    plantes = ", ".join(p["id"] for p in best_run["meilleur"]["plantes"])
    print(f"   {e:10s} → {best_run['meilleur']['score']:7.2f} : {plantes}")

**Comment lire** :
- Des **barres d'erreur courtes** = l'AG converge vers la même qualité quel que soit le tirage → solutions fiables
- Des **potagers identiques** entre environnements = solution universelle ; des potagers **différents** = chaque environnement a ses spécialistes (c'est le signe que tes contraintes de saison/substrat/fonts travaillent)
- Le **bac hiver vide** est le résultat le plus instructif : ta contrainte de saison (dure) élimine 100 % du pool. Pour le remplir : ajouter des plantes d'hiver (mâche, épinard, poireau d'hiver…) dans `plantes.json` — c'est le prochain enrichissement de données le plus utile

## 5. Comment itérer

- **Change une chose à la fois** : une valeur du bac, la seed, ou une plante dans `plantes.json`
- **Compare les runs** : si le gagnant change du tout au tout entre deux seeds, l'AG explore mal (population trop petite, early stopping trop pressé)
- **Lis le flop** : les mauvaises propositions sont aussi instructives — une compétition pénalisée, un bac quasi vide, un rendement faible
- Les **poids** (5 / 7 / 10) vivent dans `fitness/eval.rs` : changer le `7` en `12` valorise le compagnonnage, puis relance et compare les histogrammes